In [1]:
import pandas as pd

In [84]:
data = pd.read_csv('./../tmpsnx71rnm.csv')
data.columns

Index(['Unnamed: 0', 'Rank (Borda)', 'Model', 'Zero-shot',
       'Active Parameters (B)', 'Total Parameters (B)', 'Embedding Dimensions',
       'Max Tokens', 'Mean (Task)', 'Mean (TaskType)', 'Bitext Mining',
       'Classification', 'Clustering', 'Instruction Reranking',
       'Multilabel Classification', 'Pair Classification', 'Reranking',
       'Retrieval', 'STS'],
      dtype='str')

In [85]:
data.drop(columns=['Unnamed: 0', 'Zero-shot','Active Parameters (B)','Classification', 'Clustering', 'Instruction Reranking',
       'Multilabel Classification', 'Pair Classification', 'STS', 'Mean (Task)', 'Mean (TaskType)', 'Bitext Mining'], inplace=True)

In [86]:
data['Model'] = data['Model'].apply(lambda x: x.split('(')[0].strip().replace('[', '').replace(']', ''))

In [87]:
data.isnull().sum()

Rank (Borda)              0
Model                     0
Total Parameters (B)     48
Embedding Dimensions     16
Max Tokens               35
Reranking               237
Retrieval               234
dtype: int64

In [88]:
for col in data.columns:
    data[col] = data[col].fillna('0.00')
    if data[col].dtype == 'object':
        data[col]=data[col].astype(float)


In [93]:
filtered = data[
    (data['Total Parameters (B)'] != 0.00) &
    (data['Total Parameters (B)'] < 0.25)# & (data['Total Parameters (B)'] < 2.0)
]
sorted_data = filtered.sort_values('Total Parameters (B)', ascending=True)


In [94]:
sorted_data= sorted_data.sort_values('Retrieval', ascending=False)

In [95]:
sorted_data.head(10)

,Rank (Borda),Model,Total Parameters (B),Embedding Dimensions,Max Tokens,Reranking,Retrieval
19,19,jina-embeddings-v5-text-nano,0.212,768.0,8192.0,64.63,63.26
82,83,granite-embedding-97m-multilingual-r2,0.097,384.0,8192.0,59.39,60.32
57,58,F2LLM-v2-160M,0.159,640.0,40960.0,60.34,54.08
65,66,multilingual-e5-small,0.118,384.0,512.0,60.43,50.91
68,69,F2LLM-v2-80M,0.080,320.0,40960.0,58.95,50.13
59,60,bilingual-embedding-small,0.118,384.0,512.0,59.31,49.55
83,84,granite-embedding-107m-multilingual,0.107,384.0,512.0,58.48,48.08
113,114,static-similarity-mrl-multilingual-v1,0.108,1024.0,0.0,49.45,41.21
87,88,nomic-embed-text-v1-ablated,0.137,768.0,8192.0,45.04,40.74
98,99,nomic-embed-text-v1-unsupervised,0.137,768.0,8192.0,48.18,40.66


In [91]:
sorted_data.to_csv('./../filtered_sorted_models.csv', index=False)

In [3]:
import os
os.chdir('./..')

In [4]:
from src.utils import log, CustomException
log = log()

1. Based on research I chose bge-base-en-v1.5.
Features:
    - 0.1B parameters 
    - model size is 430MB approx.
    - 512 MAX tokens
    - Retriever effieciency is also good.

In [5]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('BAAI/bge-base-en-v1.5')
tokenizer = model.tokenizer

/home/vraj/.conda/envs/eu-mdr/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11918.19it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
text = "What is the capital of France? The capital of France is Paris. You can also visit the Eiffel Tower in Paris."
token_count = tokenizer.encode(text)
print(f"Number of tokens: {len(token_count)}")

Number of tokens: 28


In [7]:
import sys
import json
try:
    log.info("Loading cleaned EU MDR 2017-745 documents for token length analysis.")
    with open('data/processed/cleaned_eu_mdr_2017-745.json', 'r') as f:
        docs = json.load(f)
    for i, doc in enumerate(docs):
        token_length = len(tokenizer.encode(doc.get('page_content')))
        print(f"Token length of document {i+1}: {token_length}")
    
    
    
except Exception as e:
    log.exception(f"An error occurred: {e}")
    raise CustomException(e, sys)

2026-06-02 05:42:19,615 53 3204945777 - INFO - Loading cleaned EU MDR 2017-745 documents for token length analysis.
Token indices sequence length is longer than the specified maximum sequence length for this model (11358 > 512). Running this sequence through the model will result in indexing errors


Token length of document 1: 11358
Token length of document 2: 7
Token length of document 3: 2040
Token length of document 4: 3902
Token length of document 5: 79
Token length of document 6: 334
Token length of document 7: 31
Token length of document 8: 641
Token length of document 9: 268
Token length of document 10: 169
Token length of document 11: 259
Token length of document 12: 263
Token length of document 13: 1750
Token length of document 14: 730
Token length of document 15: 180
Token length of document 16: 802
Token length of document 17: 750
Token length of document 18: 651
Token length of document 19: 859
Token length of document 20: 1071
Token length of document 21: 462
Token length of document 22: 260
Token length of document 23: 295
Token length of document 24: 262
Token length of document 25: 667
Token length of document 26: 149
Token length of document 27: 48
Token length of document 28: 31
Token length of document 29: 111
Token length of document 30: 96
Token length of docu

In [ ]:
for doc in docs:
    doc.get('page_content', '')

list